# PAIR pipeline runner

One `run()` call = embed -> train -> validate for a given backbone. Streams
subprocess output live.

In [1]:
import re
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "code":
    REPO_ROOT = REPO_ROOT.parent

PYTHON = sys.executable
EMBED_SCRIPT    = REPO_ROOT / "code" / "data_prep" / "embed_items.py"
TRAIN_SCRIPT    = REPO_ROOT / "code" / "modelling" / "model_training.py"
VALIDATE_SCRIPT = REPO_ROOT / "code" / "modelling" / "model_validation.py"

In [ ]:
def sh(cmd):
    """Run cmd, stream output live, return it as one string."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{cmd[1]} exited with code {proc.returncode}")
    return "".join(lines)


def run(model, backend="hf", skip_embed=False, no_plots=True, **kwargs):
    """Embed (unless skip_embed) -> train -> validate one backbone.

    model:    HF id ("Qwen/Qwen3-Embedding-8B"), Ollama tag, or API model name.
    backend:  "hf" | "ollama" | "api"
    kwargs:   passed straight through as extra CLI flags to embed_items.py
              (e.g. quantize=4, batch_size=16, dims=1024).
    """
    plot_flag = ["--no-plots"] if no_plots else []
    extra = [f"--{k.replace('_', '-')}" for k, v in kwargs.items() if v is True]
    extra += [x for k, v in kwargs.items() if v is not True for x in (f"--{k.replace('_', '-')}", str(v))]

    if skip_embed:
        model_safe = model.replace(":", "-").replace("/", "-")
    else:
        out = sh([PYTHON, str(EMBED_SCRIPT), "--backend", backend, "--model", model,
                  "--splits", "train", "holdout", "validation", *extra])
        model_safe = re.findall(r"model_safe\s*=\s*(\S+)", out)[-1]

    sh([PYTHON, str(TRAIN_SCRIPT), "--emb-model", model_safe, *plot_flag])
    sh([PYTHON, str(VALIDATE_SCRIPT), "--emb-model", model_safe, *plot_flag])
    print(f"\nDone: {model_safe}")


## Use it

Call `run()` per backbone. Loop over a list if sweeping, e.g.: 

models = ["model1", "model2"]

for model in models:

    run(model = model, ...)

In [3]:

run(model="dwulff/mpnet-personality", backend="hf", skip_embed=False, no_plots=True)

$ /storage/homefs/am26g599/envs/py314/bin/python /storage/homefs/am26g599/ItemStats/code/data_prep/embed_items.py --backend hf --model dwulff/mpnet-personality --splits train holdout validation

=== Split: train ===
2,843 unique items
[get_embeddings_HF] no case for 'dwulff/mpnet-personality', using plain defaults.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 771.02it/s]

Batches: 100%|██████████| 89/89 [00:03<00:00, 24.55it/s]
Wrote (2843, 769) -> /storage/homefs/am26g599/ItemStats/data/raw/dwulff-mpnet-personality/embeddings_raw.parquet

=== Split: holdout_ ===
418 unique items
[get_embeddings_HF] no case for 'dwulff/mpnet-personality', using plain defaults.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4319.50it/s]

Batches: 100%|██████████| 14/14 [00:00<00:00, 54.27it/s]
Wrote (418, 769) -> /storage/homefs/am26g599/ItemStats/data/raw/dwulff-mpnet-personality/holdout_embeddings_raw.parquet

=== Split: validation_ ===
260 unique items
[get_embeddings_HF] no cas

'dwulff-mpnet-personality'